# HWK 7 Part II – 3-D Distribution of Bright Stars
Yale Bright Star Catalog — Cartesian Equatorial Frame with Plotly

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

### Load data

In [2]:
df = pd.read_csv('bsc5.csv')

### Drop those w/o trigonometric parallax

In [3]:
df = df.dropna(subset=['Parallax'])

df = df[df['Parallax'] > 0].copy()

### Get Distance in parsecs

In [4]:
df['Distance'] = 1.0 / df['Parallax']

### Convert RA & Dec

x = r · cos(Dec) · cos(RA)

y = r · cos(Dec) · sin(RA)

z = r · sin(Dec)

In [5]:
# RA in decimal degrees
df['ra_deg'] = (df['RAh'] + df['RAm'] / 60.0 + df['RAs'] / 3600.0) * 15.0

# Dec in decimal degrees
sign = df['DE-'].apply(lambda s: -1 if str(s).strip() == '-' else +1)
df['dec_deg'] = sign * (df['DEd'] + df['DEm'] / 60.0 + df['DEs'] / 3600.0)

# Radians
ra_rad  = np.deg2rad(df['ra_deg'])
dec_rad = np.deg2rad(df['dec_deg'])
r       = df['Distance']

# Cartesian
df['x'] = r * np.cos(dec_rad) * np.cos(ra_rad)
df['y'] = r * np.cos(dec_rad) * np.sin(ra_rad)
df['z'] = r * np.sin(dec_rad)

### Colors by spectral type

In [6]:
spectral_colors = {
    'O': '#6600FF',
    'B': '#4477FF',
    'A': '#99CCFF',
    'F': '#00CC44',
    'G': '#FFEE00',
    'K': '#FF8800',
    'M': '#FF2200',
}

def get_color(sp):
    if pd.isna(sp) or str(sp).strip() == '':
        return 'black'
    first = str(sp).strip()[0].upper()
    return spectral_colors.get(first, 'black')

df['Color'] = df['SpType'].apply(get_color)

### 3D plot

In [8]:
hover_text = [
    f"HR: {int(row.HR) if not pd.isna(row.HR) else 'N/A'}<br>"
    f"Distance: {row.Distance:.2f} pc<br>"
    f"Radial Velocity: {row.RadVel if not pd.isna(row.RadVel) else 'N/A'} km/s"
    for row in df.itertuples()
]

sp_label_map = {
    '#6600FF': 'O',
    '#4477FF': 'B',
    '#99CCFF': 'A',
    '#00CC44': 'F',
    '#FFEE00': 'G',
    '#FF8800': 'K',
    '#FF2200': 'M',
    'black':   'Other',
}

traces = []
for color, group in df.groupby('Color'):
    hover = [
        f"HR: {int(row.HR) if not pd.isna(row.HR) else 'N/A'}<br>"
        f"Distance: {row.Distance:.2f} pc<br>"
        f"Radial Velocity: {row.RadVel if not pd.isna(row.RadVel) else 'N/A'} km/s"
        for row in group.itertuples()
    ]
    traces.append(
        go.Scatter3d(
            x=group['x'],
            y=group['y'],
            z=group['z'],
            mode='markers',
            marker=dict(
                size=3,
                color=color,
                opacity=0.8,
                line=dict(width=0),
            ),
            text=hover,
            hoverinfo='text',
            name=f'Type {sp_label_map.get(color, color)}',
        )
    )

layout = go.Layout(
    title=dict(
        text='Distribution of Bright Stars',
        x=0.5, xanchor='center',
    ),
    paper_bgcolor='#0d0d1a',
    scene=dict(
        bgcolor='#0d0d1a',
        xaxis=dict(title='X (pc) → Vernal Equinox',  backgroundcolor='#0d0d1a',
                   gridcolor='#333355', color='white'),
        yaxis=dict(title='Y (pc) → RA = 6h',         backgroundcolor='#0d0d1a',
                   gridcolor='#333355', color='white'),
        zaxis=dict(title='Z (pc) → North Celestial Pole', backgroundcolor='#0d0d1a',
                   gridcolor='#333355', color='white'),
    ),
    legend=dict(
        title=dict(text='Spectral Type', font=dict(color='white')),
        font=dict(color='white'),
        bgcolor='rgba(0,0,0,0)',
    ),
    font=dict(color='white'),
    margin=dict(l=0, r=0, b=0, t=60),
    width=1000,
    height=750,
)

fig = go.Figure(data=traces, layout=layout)

fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers',
    marker=dict(size=6, color='white', symbol='circle'),
    name='Sun (origin)',
    hovertext=['Sun (0, 0, 0) pc'],
    hoverinfo='text',
))

fig.show()
#fig.write_html('3d_star_distribution.html')